In [2]:
# Cell 01 | Initialize the persistent RAG components
# Purpose: Reuse the verified SQLite-backed RAG setup for the function-calling lesson.
# Key points: This opens the existing FAQ index without re-ingestion and connects it to RAGBase.
# Execution: Run from the notebooks directory. The existing faq.db should contain 139 documents.

from dotenv import load_dotenv
from openai import OpenAI
from sqlitesearch import TextSearchIndex

from rag_helper import RAGBase


load_dotenv("../.env")

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db",
)

client = OpenAI()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=client,
)

print("Documents available:", sqlite_index.count())
print("Persistent RAG assistant ready.")

Documents available: 139
Persistent RAG assistant ready.


In [4]:
# Cell 02 | Define the search tool function
# Purpose: Expose the verified RAGBase retrieval behavior as a simple function that an LLM can call later.
# Key points: The function delegates to assistant.search() so the existing retrieval configuration remains unchanged.
# Execution: Run after Cell 01, then verify that the function returns relevant FAQ documents.

def search(query):
    """Search the LLM Zoomcamp FAQ using the verified project retrieval configuration."""
    return assistant.search(query)


test_query = "Can I still join the course after it started?"

search_results = search(test_query)

print("Results returned:", len(search_results))

for rank, doc in enumerate(search_results, start=1):
    print(f"\nRank {rank}")
    print("Section:", doc["section"])
    print("Question:", doc["question"])

Results returned: 5

Rank 1
Section: General Course-Related Questions
Question: I just discovered the course. Can I still join?

Rank 2
Section: General Course-Related Questions
Question: The homework submission form is still open even though the deadline has passed — can I still submit?

Rank 3
Section: General Course-Related Questions
Question: I missed the first homework - can I still get a certificate?

Rank 4
Section: General Course-Related Questions
Question: Certificate: Can I follow the course in a self-paced mode and get a certificate?

Rank 5
Section: General Course-Related Questions
Question: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?


In [5]:
# Cell 03 | Define the search tool schema
# Purpose: Describe the Python search() function in a format the LLM can understand.
# Key points: The schema defines the tool name, purpose, and required input arguments.
# Execution: Run this cell to create the tool definition. No API call is made yet.

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the LLM Zoomcamp FAQ for information relevant to the user's question.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to use when retrieving relevant FAQ documents.",
            }
        },
        "required": ["query"],
        "additionalProperties": False,
    },
    "strict": True,
}

print(search_tool)

{'type': 'function', 'name': 'search', 'description': "Search the LLM Zoomcamp FAQ for information relevant to the user's question.", 'parameters': {'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'The search query to use when retrieving relevant FAQ documents.'}}, 'required': ['query'], 'additionalProperties': False}, 'strict': True}


In [6]:
# Cell 04 | Request a function call from the model
# Purpose: Give the search tool definition to the LLM and inspect the requested function call.
# Key points: The model requests a tool call but does not execute the Python function itself.
# Execution: Run once and inspect the function name, arguments, and call ID.

question = "Can I still join the course after it started?"

response = client.responses.create(
    model=assistant.model,
    input=question,
    tools=[search_tool],
    tool_choice="required",
)

function_calls = [
    item
    for item in response.output
    if item.type == "function_call"
]

print("Function calls returned:", len(function_calls))

for call in function_calls:
    print("\nType:", call.type)
    print("Name:", call.name)
    print("Arguments:", call.arguments)
    print("Call ID:", call.call_id)

Function calls returned: 1

Type: function_call
Name: search
Arguments: {"query":"Can I still join the LLM Zoomcamp course after it has started? Late enrollment, registration, deadlines, catching up."}
Call ID: call_ws5sh24SFJakNbxryFC1ttPi


In [7]:
# Cell 05 | Parse the function-call arguments
# Purpose: Convert the model-generated JSON arguments into a Python dictionary.
# Key points: Parsing prepares the arguments for local Python execution.
# Execution: Run after Cell 04 and inspect the parsed query. No search is executed yet.

import json


call = function_calls[0]

arguments = json.loads(call.arguments)

print("Tool name:", call.name)
print("Arguments type:", type(arguments))
print("Parsed arguments:", arguments)
print("Search query:", arguments["query"])

Tool name: search
Arguments type: <class 'dict'>
Parsed arguments: {'query': 'Can I still join the LLM Zoomcamp course after it has started? Late enrollment, registration, deadlines, catching up.'}
Search query: Can I still join the LLM Zoomcamp course after it has started? Late enrollment, registration, deadlines, catching up.


In [8]:
# Cell 06 | Execute the search tool locally
# Purpose: Execute the model-requested search using the project's verified SQLite retrieval.
# Key points: GPT requested the tool call, but Python performs the actual retrieval.
# Execution: Run after Cell 05 and inspect the retrieved FAQ documents.

tool_result = search(**arguments)

print("Tool executed:", call.name)
print("Results returned:", len(tool_result))

for rank, doc in enumerate(tool_result, start=1):
    print(f"\nRank {rank}")
    print("Section:", doc["section"])
    print("Question:", doc["question"])

Tool executed: search
Results returned: 5

Rank 1
Section: General Course-Related Questions
Question: Where can I track the LLM Zoomcamp syllabus, deadlines, homework, and progress?

Rank 2
Section: General Course-Related Questions
Question: I just discovered the course. Can I still join?

Rank 3
Section: General Course-Related Questions
Question: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?

Rank 4
Section: General Course-Related Questions
Question: Where is the LLM Zoomcamp Telegram channel?

Rank 5
Section: Capstone Project
Question: Where can I find previous LLM Zoomcamp projects?


In [9]:
# Cell 07 | Build the function-call output
# Purpose: Package the local search result so it can be returned to the model.
# Key points: The call_id links this result to the model's original function call.
# Execution: Run after Cell 06. This cell does not make another API request.

tool_output_json = json.dumps(
    tool_result,
    ensure_ascii=False,
)

function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": tool_output_json,
}

print("Output type:", function_call_output["type"])
print("Call ID:", function_call_output["call_id"])
print("Output data type:", type(function_call_output["output"]))
print("Output preview:")
print(function_call_output["output"][:500])

Output type: function_call_output
Call ID: call_ws5sh24SFJakNbxryFC1ttPi
Output data type: <class 'str'>
Output preview:
[{"id": "20c5a1347e", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "Where can I track the LLM Zoomcamp syllabus, deadlines, homework, and progress?", "answer": "Use the [LLM Zoomcamp course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nIt contains the current cohort structure, homework, deadlines, and progress tracking. The process is the same as in other DataTalks.Club Zoomcamps."}, {"id": "74eb249bbf", "course": "llm-zoomcam


In [10]:
# Cell 08 | Return the tool output and get the final answer
# Purpose: Send the completed function call back to the model and generate a grounded answer.
# Key points: Preserve the full first-response output, then append the matching function_call_output.
# Execution: Run after Cell 07. This makes the second OpenAI API request.

message_history = [
    {
        "role": "user",
        "content": question,
    }
]

# Preserve everything the model produced in the first response,
# including the function call and any reasoning-related output items.
message_history.extend(response.output)

# Return the local tool execution result to the matching function call.
message_history.append(function_call_output)

final_response = client.responses.create(
    model=assistant.model,
    input=message_history,
    tools=[search_tool],
)

print("Final answer:")
print(final_response.output_text)

Final answer:
Yes, you can still join after the course has started. You can begin learning and submit homework while the forms are open.

To receive a certificate, make sure you submit the required project before the project-submission deadline. Check the [course platform](https://courses.datatalks.club/llm-zoomcamp-2026/) for current deadlines and progress tracking.


In [11]:
# Cell 09 | Inspect the final response structure
# Purpose: Verify that the second model response completed with a final message.
# Key points: This confirms that the single-tool round trip ended after one search call.
# Execution: Run after Cell 08 and inspect the output item types.

print("Final response output items:")

for index, item in enumerate(final_response.output, start=1):
    print(f"{index}. Type: {item.type}")

new_function_calls = [
    item
    for item in final_response.output
    if item.type == "function_call"
]

print("\nNew function calls:", len(new_function_calls))

Final response output items:
1. Type: message

New function calls: 0
